# SWE-Bench Test Dataset Patch Diff 分析

本notebook用于分析swebench_test.csv文件中每个实例的patch_diff变更的代码行数。

## 1. 导入必要的库

In [ ]:
import os
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体和编码
import matplotlib
matplotlib.rcParams['font.family'] = ['DejaVu Sans', 'SimHei', 'Arial Unicode MS', 'Heiti TC', 'PingFang SC']
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'SimHei', 'Arial Unicode MS', 'Heiti TC', 'PingFang SC']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 10

# 设置pandas显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# 设置图表样式
sns.set_style("whitegrid")
plt.style.use('default')

# 确保中文正常显示
try:
    # 尝试设置中文字体
    from matplotlib.font_manager import FontProperties
    import platform
    
    system = platform.system()
    if system == 'Darwin':  # macOS
        plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti TC', 'Arial Unicode MS']
    elif system == 'Windows':
        plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
    else:  # Linux
        plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'WenQuanYi Micro Hei', 'Arial Unicode MS']
    
    print(f"当前系统: {system}，已设置中文字体")
except Exception as e:
    print(f"字体设置警告: {e}")
    print("如果图表中文显示异常，请安装相应的中文字体")

## 2. 定义工具函数

In [ ]:
def find_git_root(start_path=None):
    """
    通过向上查找 .git 目录来确定 Git 仓库根目录
    """
    if start_path is None:
        start_path = os.getcwd()

    current_path = Path(start_path).resolve()

    # 向上遍历目录
    for parent in [current_path] + list(current_path.parents):
        git_dir = parent / '.git'
        if git_dir.exists():
            return str(parent)

    raise Exception("未找到 Git 仓库根目录")


def parse_diff_lines(diff_text):
    """
    解析git diff文本，统计变更行数
    
    Args:
        diff_text (str): git diff格式的文本
    
    Returns:
        dict: 包含添加、删除、总变更行数的字典
    """
    if not diff_text or pd.isna(diff_text):
        return {'added': 0, 'deleted': 0, 'total': 0, 'net': 0}
    
    lines = diff_text.split('\n')
    added_lines = 0
    deleted_lines = 0
    
    for line in lines:
        # 跳过diff头部信息
        if (line.startswith('diff --git') or 
            line.startswith('index ') or 
            line.startswith('--- ') or 
            line.startswith('+++ ') or 
            line.startswith('@@')):
            continue
        
        # 统计添加的行（以+开头，但不是+++）
        if line.startswith('+') and not line.startswith('+++'):
            added_lines += 1
        # 统计删除的行（以-开头，但不是---）
        elif line.startswith('-') and not line.startswith('---'):
            deleted_lines += 1
    
    total_changes = added_lines + deleted_lines
    net_changes = added_lines - deleted_lines
    
    return {
        'added': added_lines,
        'deleted': deleted_lines,
        'total': total_changes,
        'net': net_changes
    }


def analyze_patch_complexity(diff_text):
    """
    分析patch的复杂度
    
    Args:
        diff_text (str): git diff格式的文本
    
    Returns:
        dict: 包含文件数量、函数变更等信息的字典
    """
    if not diff_text or pd.isna(diff_text):
        return {'files_changed': 0, 'has_function_changes': False}
    
    # 统计变更的文件数量
    file_pattern = r'diff --git a/(.*?) b/'
    files_changed = len(re.findall(file_pattern, diff_text))
    
    # 检查是否有函数定义的变更
    function_patterns = [
        r'[+-]\s*def\s+\w+',  # Python函数
        r'[+-]\s*function\s+\w+',  # JavaScript函数
        r'[+-]\s*class\s+\w+',  # 类定义
    ]
    
    has_function_changes = any(re.search(pattern, diff_text) for pattern in function_patterns)
    
    return {
        'files_changed': files_changed,
        'has_function_changes': has_function_changes
    }


def translate_problem_statement(text):
    """
    翻译问题描述文本（简单的关键词翻译）
    
    Args:
        text (str): 英文问题描述
    
    Returns:
        str: 翻译后的中文描述
    """
    if not text or pd.isna(text) or text.strip() == '':
        return '无问题描述'
    
    # 简单的关键词翻译映射
    translation_map = {
        'bug': '错误',
        'error': '错误',
        'issue': '问题',
        'problem': '问题',
        'fix': '修复',
        'function': '函数',
        'method': '方法',
        'class': '类',
        'module': '模块',
        'import': '导入',
        'exception': '异常',
        'TypeError': '类型错误',
        'ValueError': '值错误',
        'AttributeError': '属性错误',
        'KeyError': '键错误',
        'IndexError': '索引错误',
        'NameError': '名称错误',
        'test': '测试',
        'testing': '测试',
        'fail': '失败',
        'failure': '失败',
        'pass': '通过',
        'return': '返回',
        'parameter': '参数',
        'argument': '参数',
        'variable': '变量',
        'deprecated': '已弃用',
        'warning': '警告',
        'documentation': '文档',
        'example': '示例',
        'expected': '期望的',
        'actual': '实际的',
        'should': '应该',
        'when': '当',
        'if': '如果',
        'then': '那么',
        'but': '但是',
        'however': '然而',
        'instead': '相反',
        'currently': '目前',
        'now': '现在',
        'before': '之前',
        'after': '之后'
    }
    
    # 截取前200个字符以避免过长
    text = text[:200] if len(text) > 200 else text
    
    # 简单的关键词替换
    translated = text.lower()
    for en_word, zh_word in translation_map.items():
        translated = translated.replace(en_word, zh_word)
    
    # 如果翻译后的文本与原文差异很小，说明翻译效果不好，返回原文摘要
    if len(set(translated.split()) & set(text.lower().split())) > len(translated.split()) * 0.8:
        # 返回原文的前100个字符作为摘要
        return f"[英文] {text[:100]}{'...' if len(text) > 100 else ''}"
    
    return translated.capitalize()

## 3. 加载数据

In [ ]:
# 加载数据
try:
    csv_path = find_git_root() + '/swebench_test.csv'
    print(f"正在加载数据: {csv_path}")
    
    # 使用与run_infer.py相同的方式加载数据
    instances = pd.read_csv(csv_path, encoding='utf-8', dtype=str).astype(str).replace('nan', '')
    
    print(f"成功加载 {len(instances)} 个实例")
    print(f"数据列: {list(instances.columns)}")
    print(f"数据形状: {instances.shape}")
    
except Exception as e:
    print(f"加载数据时出错: {e}")
    # 如果无法加载真实数据，创建示例数据用于演示
    print("使用示例数据进行演示...")
    
    sample_data = {
        'repo': ['astropy/astropy', 'django/django', 'scikit-learn/scikit-learn'],
        'instance_id': ['astropy__astropy-12907', 'django__django-12345', 'sklearn__sklearn-67890'],
        'base_commit': ['d16bfe05a744909de4b27f5875fe0d4ed41ce607', 'abc123', 'def456'],
        'patch': [
            '''diff --git a/astropy/modeling/separable.py b/astropy/modeling/separable.py\n--- a/astropy/modeling/separable.py\n+++ b/astropy/modeling/separable.py\n@@ -242,7 +242,7 @@ def _cstack(left, right):\n         cright = _coord_matrix(right, 'right', noutp)\n     else:\n         cright = np.zeros((noutp, right.shape[1]))\n-        cright[-right.shape[0]:, -right.shape[1]:] = 1\n+        cright[-right.shape[0]:, -right.shape[1]:] = right\n \n     return np.hstack([cleft, cright])''',
            '''diff --git a/django/forms/fields.py b/django/forms/fields.py\n--- a/django/forms/fields.py\n+++ b/django/forms/fields.py\n@@ -100,6 +100,8 @@ class Field:\n         self.required = required\n         self.widget = widget\n         self.label = label\n+        self.help_text = help_text\n+        self.initial = initial\n         \n     def validate(self, value):\n-        if self.required and not value:\n+        if self.required and value in self.empty_values:\n             raise ValidationError(self.error_messages['required'])''',
            '''diff --git a/sklearn/linear_model/base.py b/sklearn/linear_model/base.py\n--- a/sklearn/linear_model/base.py\n+++ b/sklearn/linear_model/base.py\n@@ -50,10 +50,12 @@ class LinearModel:\n     def fit(self, X, y):\n         X, y = check_X_y(X, y)\n         \n-        # Simple implementation\n-        self.coef_ = np.linalg.lstsq(X, y, rcond=None)[0]\n+        # Improved implementation with regularization\n+        if self.alpha > 0:\n+            self.coef_ = np.linalg.solve(X.T @ X + self.alpha * np.eye(X.shape[1]), X.T @ y)\n+        else:\n+            self.coef_ = np.linalg.lstsq(X, y, rcond=None)[0]\n         \n         return self'''
        ],
        'test_patch': ['', '', '']
    }
    
    instances = pd.DataFrame(sample_data)
    print(f"使用示例数据，包含 {len(instances)} 个实例")

# 如果存在problem_statement列，进行翻译处理
if 'problem_statement' in instances.columns:
    print("\n正在翻译problem_statement列...")
    instances['problem_statement_zh'] = instances['problem_statement'].apply(translate_problem_statement)
    print(f"翻译完成，共处理 {len(instances)} 个问题描述")
    
    # 显示翻译示例
    print("\n翻译示例:")
    for i in range(min(3, len(instances))):
        original = instances.iloc[i]['problem_statement'][:100] + '...' if len(instances.iloc[i]['problem_statement']) > 100 else instances.iloc[i]['problem_statement']
        translated = instances.iloc[i]['problem_statement_zh']
        print(f"\n实例 {i+1}:")
        print(f"原文: {original}")
        print(f"译文: {translated}")
else:
    print("\n未找到problem_statement列，跳过翻译步骤")

## 4. 分析patch变更

In [ ]:
# 分析每个实例的patch变更
print("正在分析patch变更...")

# 初始化结果列表
analysis_results = []

for idx, row in instances.iterrows():
    # 分析主要patch
    patch_stats = parse_diff_lines(row.get('patch', ''))
    complexity_stats = analyze_patch_complexity(row.get('patch', ''))
    
    # 分析测试patch（如果存在）
    test_patch_stats = parse_diff_lines(row.get('test_patch', ''))
    
    result = {
        'repo': row.get('repo', ''),
        'instance_id': row.get('instance_id', ''),
        'base_commit': row.get('base_commit', ''),
        
        # 主要patch统计
        'patch_added_lines': patch_stats['added'],
        'patch_deleted_lines': patch_stats['deleted'],
        'patch_total_changes': patch_stats['total'],
        'patch_net_changes': patch_stats['net'],
        
        # 测试patch统计
        'test_patch_added_lines': test_patch_stats['added'],
        'test_patch_deleted_lines': test_patch_stats['deleted'],
        'test_patch_total_changes': test_patch_stats['total'],
        
        # 复杂度统计
        'files_changed': complexity_stats['files_changed'],
        'has_function_changes': complexity_stats['has_function_changes'],
        
        # 总体统计
        'total_added_lines': patch_stats['added'] + test_patch_stats['added'],
        'total_deleted_lines': patch_stats['deleted'] + test_patch_stats['deleted'],
        'total_changes': patch_stats['total'] + test_patch_stats['total']
    }
    
    analysis_results.append(result)

# 转换为DataFrame
df_analysis = pd.DataFrame(analysis_results)

print(f"分析完成，共处理 {len(df_analysis)} 个实例")
print("\n前5个实例的分析结果:")
display(df_analysis.head())

## 5. 基础统计信息

In [ ]:
# 基础统计信息
print("=== 基础统计信息 ===")
print(f"总实例数: {len(df_analysis)}")
print(f"涉及的仓库数: {df_analysis['repo'].nunique()}")
print(f"有代码变更的实例数: {(df_analysis['patch_total_changes'] > 0).sum()}")
print(f"有测试变更的实例数: {(df_analysis['test_patch_total_changes'] > 0).sum()}")
print(f"有函数级变更的实例数: {df_analysis['has_function_changes'].sum()}")

print("\n=== 代码变更行数统计 ===")
stats_columns = ['patch_added_lines', 'patch_deleted_lines', 'patch_total_changes', 
                'test_patch_total_changes', 'total_changes']

stats_df = df_analysis[stats_columns].describe()
display(stats_df)

print("\n=== 按仓库统计 ===")
repo_stats = df_analysis.groupby('repo').agg({
    'instance_id': 'count',
    'patch_total_changes': ['mean', 'median', 'max'],
    'total_changes': ['mean', 'median', 'max'],
    'files_changed': 'mean'
}).round(2)

repo_stats.columns = ['实例数', '平均patch变更', '中位数patch变更', '最大patch变更',
                     '平均总变更', '中位数总变更', '最大总变更', '平均文件数']
display(repo_stats)

print("\n=== 每个仓库的每个实例详细变更信息 ===")
# 按仓库分组显示每个实例的详细信息
for repo in df_analysis['repo'].unique():
    repo_data = df_analysis[df_analysis['repo'] == repo]
    print(f"\n仓库: {repo}")
    print(f"实例总数: {len(repo_data)}")
    
    # 显示该仓库下每个实例的变更信息
    instance_details = repo_data[[
        'instance_id', 'patch_added_lines', 'patch_deleted_lines', 
        'patch_total_changes', 'files_changed'
    ]].sort_values('patch_total_changes', ascending=False)
    
    instance_details.columns = ['实例ID', '添加行数', '删除行数', '总变更行数', '变更文件数']
    display(instance_details)
    
    # 该仓库的统计摘要
    print(f"该仓库统计摘要:")
    print(f"  - 平均变更行数: {repo_data['patch_total_changes'].mean():.2f}")
    print(f"  - 中位数变更行数: {repo_data['patch_total_changes'].median():.2f}")
    print(f"  - 最大变更行数: {repo_data['patch_total_changes'].max()}")
    print(f"  - 最小变更行数: {repo_data['patch_total_changes'].min()}")
    print(f"  - 变更行数标准差: {repo_data['patch_total_changes'].std():.2f}")
    print("-" * 80)

## 6. 数据可视化

In [ ]:
# 创建图表
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('SWE-Bench Test Dataset Patch 变更分析', fontsize=16, fontweight='bold')

# 1. 总变更行数分布
axes[0, 0].hist(df_analysis['patch_total_changes'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Patch 总变更行数分布')
axes[0, 0].set_xlabel('变更行数')
axes[0, 0].set_ylabel('实例数量')
axes[0, 0].grid(True, alpha=0.3)

# 2. 添加 vs 删除行数散点图
axes[0, 1].scatter(df_analysis['patch_added_lines'], df_analysis['patch_deleted_lines'], 
                  alpha=0.6, color='coral')
axes[0, 1].set_title('添加行数 vs 删除行数')
axes[0, 1].set_xlabel('添加行数')
axes[0, 1].set_ylabel('删除行数')
axes[0, 1].grid(True, alpha=0.3)

# 3. 变更文件数分布
file_counts = df_analysis['files_changed'].value_counts().sort_index()
axes[0, 2].bar(file_counts.index, file_counts.values, color='lightgreen', alpha=0.7)
axes[0, 2].set_title('变更文件数分布')
axes[0, 2].set_xlabel('文件数')
axes[0, 2].set_ylabel('实例数量')
axes[0, 2].grid(True, alpha=0.3)

# 4. 按仓库的变更行数箱线图
if df_analysis['repo'].nunique() <= 10:
    df_analysis.boxplot(column='patch_total_changes', by='repo', ax=axes[1, 0])
    axes[1, 0].set_title('按仓库的变更行数分布')
    axes[1, 0].set_xlabel('仓库')
    axes[1, 0].set_ylabel('变更行数')
    plt.setp(axes[1, 0].xaxis.get_majorticklabels(), rotation=45)
else:
    top_repos = df_analysis.groupby('repo')['patch_total_changes'].mean().nlargest(10)
    axes[1, 0].bar(range(len(top_repos)), top_repos.values, color='orange', alpha=0.7)
    axes[1, 0].set_title('Top 10 仓库平均变更行数')
    axes[1, 0].set_xlabel('仓库排名')
    axes[1, 0].set_ylabel('平均变更行数')
    axes[1, 0].grid(True, alpha=0.3)

# 5. Patch vs Test Patch 变更对比
axes[1, 1].scatter(df_analysis['patch_total_changes'], df_analysis['test_patch_total_changes'], 
                  alpha=0.6, color='purple')
axes[1, 1].set_title('Patch 变更 vs Test Patch 变更')
axes[1, 1].set_xlabel('Patch 变更行数')
axes[1, 1].set_ylabel('Test Patch 变更行数')
axes[1, 1].grid(True, alpha=0.3)

# 6. 净变更行数分布
axes[1, 2].hist(df_analysis['patch_net_changes'], bins=30, alpha=0.7, color='gold', edgecolor='black')
axes[1, 2].set_title('净变更行数分布 (添加-删除)')
axes[1, 2].set_xlabel('净变更行数')
axes[1, 2].set_ylabel('实例数量')
axes[1, 2].axvline(x=0, color='red', linestyle='--', alpha=0.7, label='零变更线')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 详细分析

In [ ]:
# 找出变更最大的实例
print("=== 变更行数最多的Top 10实例 ===")
top_changes = df_analysis.nlargest(10, 'patch_total_changes')[[
    'repo', 'instance_id', 'patch_added_lines', 'patch_deleted_lines', 
    'patch_total_changes', 'files_changed'
]]
display(top_changes)

print("\n=== 变更类型分析 ===")
# 分类变更类型
def categorize_change(row):
    if row['patch_total_changes'] == 0:
        return '无变更'
    elif row['patch_added_lines'] == 0:
        return '仅删除'
    elif row['patch_deleted_lines'] == 0:
        return '仅添加'
    elif row['patch_added_lines'] > row['patch_deleted_lines']:
        return '主要添加'
    elif row['patch_deleted_lines'] > row['patch_added_lines']:
        return '主要删除'
    else:
        return '平衡变更'

df_analysis['change_type'] = df_analysis.apply(categorize_change, axis=1)
change_type_counts = df_analysis['change_type'].value_counts()
print(change_type_counts)

# 可视化变更类型
plt.figure(figsize=(10, 6))
plt.pie(change_type_counts.values, labels=change_type_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('变更类型分布')
plt.axis('equal')
plt.show()

## 8. 导出分析结果

In [ ]:
# 保存分析结果到CSV文件
output_file = 'patch_analysis_results.csv'
df_analysis.to_csv(output_file, index=False, encoding='utf-8')
print(f"分析结果已保存到: {output_file}")

# 生成摘要报告
summary_report = f"""
SWE-Bench Test Dataset Patch 分析摘要报告
==========================================

数据概览:
- 总实例数: {len(df_analysis)}
- 涉及仓库数: {df_analysis['repo'].nunique()}
- 有代码变更的实例: {(df_analysis['patch_total_changes'] > 0).sum()}
- 有测试变更的实例: {(df_analysis['test_patch_total_changes'] > 0).sum()}

变更行数统计:
- 平均patch变更行数: {df_analysis['patch_total_changes'].mean():.2f}
- 中位数patch变更行数: {df_analysis['patch_total_changes'].median():.2f}
- 最大patch变更行数: {df_analysis['patch_total_changes'].max()}
- 最小patch变更行数: {df_analysis['patch_total_changes'].min()}

变更类型分布:
{change_type_counts.to_string()}

Top 5 变更最多的仓库:
{df_analysis.groupby('repo')['patch_total_changes'].mean().nlargest(5).to_string()}
"""

print(summary_report)

# 保存摘要报告
with open('patch_analysis_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary_report)

print("\n摘要报告已保存到: patch_analysis_summary.txt")

# 为每个仓库生成单独的详细CSV文件
print("\n=== 生成按仓库分组的详细CSV文件 ===")
for repo in df_analysis['repo'].unique():
    repo_data = df_analysis[df_analysis['repo'] == repo]
    
    # 清理仓库名称用作文件名
    safe_repo_name = repo.replace('/', '_').replace(' ', '_')
    repo_file = f'patch_analysis_{safe_repo_name}.csv'
    
    # 选择关键列并按变更行数排序
    repo_details = repo_data[[
        'instance_id', 'base_commit', 'patch_added_lines', 'patch_deleted_lines',
        'patch_total_changes', 'patch_net_changes', 'files_changed', 
        'has_function_changes', 'test_patch_total_changes', 'total_changes'
    ]].sort_values('patch_total_changes', ascending=False)
    
    repo_details.to_csv(repo_file, index=False, encoding='utf-8')
    print(f"仓库 {repo} 的详细分析已保存到: {repo_file} (共{len(repo_details)}个实例)")

print("\n所有分析文件生成完成！")